# Análisis Estadístico — Auditoría E-Gov Suroccidente Guatemala

Notebook auxiliar para exploración interactiva de los resultados generados por `python main.py --all`.

**Prerrequisito:** haber corrido el pipeline al menos una vez. Esto genera `data/processed/resultados.csv`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.analysis import stats as A

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/resultados.csv')
print(f'Filas: {len(df)}, columnas: {len(df.columns)}')
df.head()

## 1. Tasa de alcance del estudio (¿cuántas URLs respondieron?)

In [ ]:
df['reachable'].fillna(False).astype(bool).value_counts(normalize=True) * 100

## 2. OE1 — Rendimiento y accesibilidad

In [ ]:
A.descriptiva_numerica(df, ['ttfb_ms', 'tiempo_total_ms', 'tamanio_kb',
                            'imgs_pct_con_alt', 'score_local_performance'])

In [ ]:
A.proporciones(df, ['https', 'tiene_viewport', 'tiene_lang', 'tiene_charset'])

## 3. OE2 — Frecuencia de actualización y LAIP

In [ ]:
A.descriptiva_numerica(df, ['snapshots_unicos', 'intervalo_medio_dias',
                            'dias_desde_ultima_actualizacion',
                            'laip_pct_cumplimiento', 'score_local_freshness'])

In [ ]:
laip_cols = ['laip_transparencia', 'laip_presupuesto', 'laip_compras',
             'laip_personal', 'laip_servicios', 'laip_estructura', 'laip_contacto']
A.proporciones(df, laip_cols)

## 4. OE3 — Seguridad básica

In [ ]:
A.descriptiva_numerica(df, ['ssl_dias_restantes', 'headers_seguridad_pct',
                            'score_local_security'])

In [ ]:
sec_bool = ['ssl_ok', 'tls_aceptable', 'cert_vigencia_suficiente',
            'redirige_a_https', 'header_hsts', 'header_csp',
            'header_x_frame_options', 'header_x_content_type_options',
            'expone_server_header']
A.proporciones(df, sec_bool)

## 5. Comparación entre departamentos (Kruskal-Wallis)
Detecta si hay diferencias estadísticamente significativas (α=0.05) entre departamentos en cada métrica.

In [ ]:
A.comparar_departamentos_kruskal(df, [
    'tiempo_total_ms', 'tamanio_kb', 'score_local_performance',
    'laip_pct_cumplimiento', 'score_local_freshness',
    'headers_seguridad_pct', 'score_local_security',
])

## 6. Correlaciones entre los 3 ejes de la investigación
¿Los portales con mejor rendimiento son también los más actualizados y seguros?

In [ ]:
corr = A.correlaciones(df, ['score_local_performance', 'score_local_freshness',
                            'score_local_security', 'laip_pct_cumplimiento',
                            'snapshots_unicos', 'headers_seguridad_pct'])
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlaciones Spearman entre los 3 ejes')
plt.tight_layout()
plt.show()